# Emulación del workflow KNIME `P&G_COCO` — modo SOLO LECTURA

Este notebook reproduce el proceso del KNIME contra el DWH (SQL Server), usando los 475 scripts
extraídos en `sql/` y el orden real de ejecución de los nodos (`plan_ejecucion.json`,
generado con `tools/build_plan.py` a partir del grafo del workflow).

**Garantías de seguridad (esto es una prueba, no debe afectar la base):**

1. Solo se ejecutan scripts que **leen** o que trabajan con **tablas temporales `#`** (viven en
   `tempdb`, son de la sesión y desaparecen al cerrar la conexión — no tocan datos permanentes).
2. Antes de ejecutar cada paso, un **validador** revisa el SQL: si detecta `INSERT/UPDATE/DELETE/`
   `CREATE/DROP/ALTER/TRUNCATE/SELECT INTO` sobre una tabla **permanente**, el paso se **omite**
   y se reporta. En todo el flujo solo hay **3** scripts así (ver auditoría más abajo).
3. Donde KNIME hacía `DB Insert` a `PL_COL_DATOS_COCO`, aquí el resultado se queda en
   **DataFrames de pandas** (y opcionalmente CSVs locales en `notebooks/salidas/`).

**Requisitos:** `pandas`, `pyodbc`, un driver ODBC de SQL Server instalado, acceso de red al DWH
y el archivo `notebooks/credenciales_local.py` (está en `.gitignore`, nunca se sube al repo).

In [ ]:
# Si falta algo, descomenta y ejecuta una vez:
# %pip install pyodbc pandas

In [ ]:
import json
import re
import time
import warnings
from pathlib import Path

import pandas as pd
import pyodbc

# --- Rutas del repo (el NB vive en notebooks/) ---
RAIZ = Path.cwd() if (Path.cwd() / 'plan_ejecucion.json').exists() else Path.cwd() / 'notebooks'
REPO = RAIZ.parent

# --- Credenciales: se leen desde credenciales_local.py (fuera de git) ---
import importlib.util
_spec = importlib.util.spec_from_file_location('credenciales_local', RAIZ / 'credenciales_local.py')
cred = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cred)
print('Credenciales cargadas para', cred.DEFAULT_USER_DWH, '@', cred.DEFAULT_HOST_DWH)

In [ ]:
# --- Conexión al DWH (SQL Server) ---
_drivers = [d for d in pyodbc.drivers() if 'SQL Server' in d]
DRIVER = sorted(_drivers, reverse=True)[0]  # prefiere ODBC Driver 18/17 si existe

CADENA = (
    f'DRIVER={{{DRIVER}}};'
    f'SERVER={cred.DEFAULT_HOST_DWH},{cred.DEFAULT_PORT_DWH};'
    f'DATABASE={cred.DEFAULT_DATABASE_DWH};'
    f'UID={cred.DEFAULT_USER_DWH};PWD={cred.DEFAULT_PASSWORD_DWH};'
)
if 'ODBC Driver' in DRIVER:
    CADENA += 'Encrypt=yes;TrustServerCertificate=yes;'

def nueva_conexion():
    """Una conexión = una sesión. Las tablas temporales # viven en la sesión,
    igual que en KNIME cada componente tiene su propio conector."""
    return pyodbc.connect(CADENA, autocommit=True, timeout=30)

with nueva_conexion() as _c:
    _cur = _c.cursor()
    _v = _cur.execute("select @@SERVERNAME, SUSER_SNAME(), DB_NAME()").fetchone()
    # ¿Tenemos acceso a la base de trabajo de actuaría? (los scripts hacen USE sobre ella)
    try:
        _cur.execute('USE Liberty_pruebas_actuaria')
        ACCESO_PRUEBAS = True
    except pyodbc.Error:
        ACCESO_PRUEBAS = False

print(f'Conectado OK — servidor: {_v[0]} | usuario: {_v[1]} | base: {_v[2]} | driver: {DRIVER}')
if not ACCESO_PRUEBAS:
    print('AVISO: tu usuario NO tiene acceso a Liberty_pruebas_actuaria.')
    print('  - Se quitará el "USE Liberty_pruebas_actuaria" de los scripts automáticamente.')
    print('  - Los componentes que SOLO leen del DWH Liberty (p. ej. Gross_Writte__43) funcionan igual.')
    print('  - Los que leen tablas PL_*/PNL_* de esa base fallarán con "Invalid object name":')
    print('    pide permiso de LECTURA sobre Liberty_pruebas_actuaria para emularlos.')

In [ ]:
# =====================  PARÁMETROS  =====================
# La única variable de flujo del KNIME (nodo Variable Expressions #70):
PERIODO_CONTABLE = '202512'   # <-- mes a procesar (YYYYMM)

# Qué componentes correr. Empieza con el más liviano para probar la conexión
# y luego amplía. None = todos los componentes CONECTADOS del flujo.
COMPONENTES_A_EJECUTAR = ['Gross_Writte__43']
# Ejemplos:
# COMPONENTES_A_EJECUTAR = None                                  # todo el flujo conectado
# COMPONENTES_A_EJECUTAR = ['Written_Prem__33', 'CHANGE_IN_CA__320']  # respeta este orden

PLAN = json.loads((RAIZ / 'plan_ejecucion.json').read_text(encoding='utf-8'))
PLAN_IDX = {c['nombre']: c for c in PLAN['componentes']}
print('Componentes disponibles:')
for c in PLAN['componentes']:
    print(f"  {'[conectado]' if c['conectado'] else '[autónomo] '} {c['nombre']:<22} {len(c['pasos'])} pasos")

In [ ]:
# =====================  HELPERS  =====================
MARCADOR = re.escape(PLAN['marcador_variable'])  # $${Speriodo_contable}$$
_RE_USE = re.compile(r'(?im)^\s*USE\s+Liberty_pruebas_actuaria\s*;?\s*$')

def cargar_sql(rel):
    """Lee un .sql del repo y sustituye la variable de flujo igual que
    KNIME (string -> literal entre comillas). Si el usuario no tiene acceso a
    Liberty_pruebas_actuaria, elimina el USE para poder correr lo que sí lee
    del DWH Liberty."""
    sql = (REPO / rel).read_text(encoding='utf-8')
    sql = re.sub(MARCADOR, f"'{PERIODO_CONTABLE}'", sql)
    if not ACCESO_PRUEBAS:
        sql = _RE_USE.sub('-- USE omitido (sin acceso a Liberty_pruebas_actuaria)', sql)
    return sql

_PELIGROS = [
    (r'\binsert\s+into\s+(?!#)', 'INSERT en tabla permanente'),
    (r'\bdelete\s+from\s+(?!#)', 'DELETE en tabla permanente'),
    (r'\btruncate\s+table\s+(?!#)', 'TRUNCATE en tabla permanente'),
    (r'\bcreate\s+table\s+(?!#)', 'CREATE TABLE permanente'),
    (r'\balter\s+table\s+(?!#)', 'ALTER TABLE permanente'),
    (r'\bmerge\b', 'MERGE'),
    (r'\bexec(ute)?\b', 'EXEC'),
]

def es_seguro(sql):
    """(ok, motivos). ok=True si el script solo lee o trabaja con temporales #."""
    body = re.sub(r'--[^\n]*', ' ', sql)      # sin comentarios
    body = re.sub(r"'[^']*'", "''", body)     # sin literales de texto
    motivos = []
    for pat, msg in _PELIGROS:
        if re.search(pat, body, re.I):
            motivos.append(msg)
    # DROP TABLE [IF EXISTS] <objetivo>: solo permitido sobre temporales #
    for m in re.finditer(r'\bdrop\s+table\s+(?:if\s+exists\s+)?(\[?[#\w.\]]+)', body, re.I):
        t = m.group(1).strip('[]')
        if not t.startswith('#'):
            motivos.append(f"DROP TABLE permanente '{t}'")
    # UPDATE: permitido solo si el destino es una temporal (directo o vía alias)
    for m in re.finditer(r'\bupdate\s+(\[?[#\w.\]]+)', body, re.I):
        t = m.group(1).strip('[]')
        if t.startswith('#'):
            continue
        if re.search(rf'\bfrom\s+#\w+\s+(?:as\s+)?{re.escape(t)}\b', body, re.I):
            continue  # alias de una temporal: update a ... from #tabla a
        motivos.append(f"UPDATE sobre '{t}' (no temporal)")
    # SELECT ... INTO permanente
    for m in re.finditer(r'\binto\s+(\[?[#\w.\]]+)', body, re.I):
        t = m.group(1).strip('[]')
        if not t.startswith('#') and not re.search(rf'insert\s+into\s+\[?{re.escape(t)}', body, re.I):
            motivos.append(f"SELECT INTO tabla permanente '{t}'")
    return (not motivos, sorted(set(motivos)))

# Auditoría previa: ¿qué pasos se van a omitir en TODO el plan?
# Esperados: 3 -> Written_Prem/DDL 197 y CHANGE_IN_CA__320/DDL 16 (reconstruyen la
# tabla de apoyo 'intermediarios_unicos') y Recobros_sin/DML 314 (INSERT permanente).
print('Pasos que el validador OMITIRÁ por escribir en tablas permanentes:')
for c in PLAN['componentes']:
    for p in c['pasos']:
        ok, motivos = es_seguro(cargar_sql(p['archivo']))
        if not ok:
            print(f"  - {p['archivo']}  ->  {motivos}")
print('(fin de la auditoría)')

In [ ]:
# =====================  MOTOR DE EJECUCIÓN  =====================
def ejecutar_componente(nombre):
    """Ejecuta los pasos de un componente en el orden real del KNIME.
    - 'executor' (DDL): construye temporales en tempdb (misma sesión).
    - 'reader'  (DML): lee a un DataFrame (lo que KNIME insertaba en PL_COL_*).
    Devuelve (dfs, omitidos)."""
    comp = PLAN_IDX[nombre]
    dfs, omitidos = {}, []
    print(f"\n=== {comp['titulo']} ({len(comp['pasos'])} pasos) ===")
    t0c = time.time()
    with nueva_conexion() as con:   # sesión nueva por componente, como en KNIME
        for i, paso in enumerate(comp['pasos'], 1):
            sql = cargar_sql(paso['archivo'])
            ok, motivos = es_seguro(sql)
            etiqueta = paso['nodo'].split('/')[-1]
            if not ok:
                omitidos.append((paso['archivo'], motivos))
                print(f'  [{i:>3}/{len(comp["pasos"])}] OMITIDO  {etiqueta}  {motivos}')
                continue
            t0 = time.time()
            try:
                if paso['tipo'] == 'executor':
                    cur = con.cursor()
                    cur.execute(sql)
                    while cur.nextset():
                        pass
                    cur.close()
                    print(f'  [{i:>3}/{len(comp["pasos"])}] DDL ok   {etiqueta}  ({time.time()-t0:,.1f}s)')
                else:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        df = pd.read_sql(sql, con)
                    dfs[paso['nodo']] = df
                    print(f'  [{i:>3}/{len(comp["pasos"])}] DML ok   {etiqueta}  {df.shape[0]:,} filas ({time.time()-t0:,.1f}s)')
            except Exception as e:
                print(f'  [{i:>3}/{len(comp["pasos"])}] ERROR    {etiqueta}: {str(e)[:200]}')
                raise
    print(f'--- {comp["titulo"]} terminado en {time.time()-t0c:,.1f}s ---')
    return dfs, omitidos

In [ ]:
# =====================  EJECUCIÓN  =====================
if COMPONENTES_A_EJECUTAR is None:
    a_correr = [c['nombre'] for c in PLAN['componentes'] if c['conectado']]
else:
    a_correr = COMPONENTES_A_EJECUTAR

resultados, omitidos_total = {}, {}
for nombre in a_correr:
    dfs, om = ejecutar_componente(nombre)
    resultados[nombre] = dfs
    if om:
        omitidos_total[nombre] = om

print('\n===== RESUMEN =====')
for nombre, dfs in resultados.items():
    filas = sum(len(d) for d in dfs.values())
    print(f'  {nombre:<22} {len(dfs)} lecturas, {filas:,} filas totales')
if omitidos_total:
    print('  Pasos omitidos (escriben en permanente):', omitidos_total)

In [ ]:
# =====================  EXPLORAR RESULTADOS  =====================
# resultados[componente][nodo] es un DataFrame. Ejemplo:
for nombre, dfs in resultados.items():
    for nodo, df in dfs.items():
        print(f'{nombre} | {nodo.split("/")[-1]:<28} {df.shape}')

# Mira uno en detalle (ajusta el índice a gusto):
if resultados:
    _comp = next(iter(resultados))
    if resultados[_comp]:
        _nodo = next(iter(resultados[_comp]))
        display(resultados[_comp][_nodo].head(10))

In [ ]:
# =====================  SALIDA CONSOLIDADA (emula PL_COL_DATOS_COCO)  =====================
# Une todas las lecturas que traen VALOR_CONCEPTO y arma el mini-P&G del periodo,
# que es exactamente lo que quedaría en la tabla si KNIME hubiera hecho los DB Insert.
piezas = []
for nombre, dfs in resultados.items():
    for nodo, df in dfs.items():
        d = df.copy()
        d.columns = [c.upper() for c in d.columns]
        if 'VALOR_CONCEPTO' in d.columns:
            d['COMPONENTE'] = nombre
            d['NODO'] = nodo.split('/')[-1]
            piezas.append(d)

if piezas:
    salida = pd.concat(piezas, ignore_index=True)
    print(f'Salida consolidada: {salida.shape[0]:,} filas')
    niveles = [c for c in ['CONCEPTO_NIVEL_0', 'CONCEPTO_NIVEL_1'] if c in salida.columns]
    if niveles:
        pyg = (salida.groupby(niveles, dropna=False)['VALOR_CONCEPTO']
               .sum().reset_index().sort_values(niveles))
        display(pyg.style.format({'VALOR_CONCEPTO': '{:,.0f}'}))
else:
    print('Aún no hay resultados con VALOR_CONCEPTO — ejecuta algún componente primero.')

In [ ]:
# =====================  EXPORTAR (opcional, solo archivos locales)  =====================
EXPORTAR = False   # pon True para guardar CSVs en notebooks/salidas/<periodo>/
if EXPORTAR and resultados:
    destino = RAIZ / 'salidas' / PERIODO_CONTABLE
    destino.mkdir(parents=True, exist_ok=True)
    for nombre, dfs in resultados.items():
        for nodo, df in dfs.items():
            archivo = re.sub(r'[^\w]+', '_', f'{nombre}__{nodo.split("/")[-1]}') + '.csv'
            df.to_csv(destino / archivo, index=False, encoding='utf-8-sig')
    print(f'CSVs guardados en {destino}')

## Diferencias frente al KNIME real (importante)

| KNIME | Este notebook |
|---|---|
| `DB SQL Executor` crea temporales en la sesión del conector | Igual: los scripts DDL corren en una conexión por componente |
| `DB Query Reader` trae el resultado a la tabla de KNIME | `pd.read_sql` → DataFrame |
| `GroupBy` / `Concatenate` / `Column Expressions` (nodos KNIME) | **No se emulan** — si un componente depende de transformaciones hechas en nodos KNIME (no SQL), el consolidado puede diferir |
| `DB Insert` escribe en `PL_COL_DATOS_COCO` / `PL_COL_DATOS` | **Nunca escribe**: los datos quedan en `resultados` y CSVs opcionales |
| Cargas Excel de la Etapa 0 (homologaciones) | No se ejecutan; se asume que las tablas `PL_*`/`PNL_*` ya existen en `Liberty_pruebas_actuaria` |
| `Recobros_sin (#315)`: `INSERT INTO PL_COL_DATOS_COCO_UNIFICADO_RC` | El validador lo **omite** automáticamente |
| `Written Prem (#33)` y `CHANGE_IN_CA (#320)` reconstruyen la tabla de apoyo permanente `intermediarios_unicos` | El validador **omite** esos 2 pasos: se usa la tabla tal como quedó de la última corrida real del KNIME (si el catálogo de intermediarios no cambió, el resultado es el mismo) |

Otros puntos:
- Los scripts hacen `USE Liberty_pruebas_actuaria`, así que tu usuario necesita permiso de **lectura**
  en esa base además de `Liberty` (crear `#temporales` en tempdb lo puede hacer cualquier usuario).
- Correr `COMPONENTES_A_EJECUTAR = None` (todo el flujo) puede tardar bastante — son consultas
  pesadas de cierre mensual. Empieza por un componente.
- El orden dentro de cada componente es el **orden topológico real** del workflow, incluidos los
  subcomponentes (`OTRAS COMISI`, `PROFIT`).